# camkit3D Record and Sync

#### Notebook to show you how to record videos from multiple cameras and then synchronise them offline

## Import modules

In [ ]:
from camkit3d.recorder import MultiCamRecorder, create_recorder
import time
from datetime import datetime

## Setup recorder


In [2]:
data_dir = '/Users/robertseymour/Documents/recordings/'

recorder = MultiCamRecorder(
    camera_ids=[0, 1],  # Your cameras
    fps=30,
    base_output_dir = data_dir
)

INFO:camkit3d.recorder:MultiCamRecorder initialized with cameras: [0, 1]


In [3]:
# Connect to cameras (once)
recorder.connect_cameras()

INFO:camkit3d.recorder:Connecting to camera 0...
INFO:camkit3d.recorder:Camera 0 (Camera 0): 1280x720 @ 30fps
INFO:camkit3d.recorder:Connecting to camera 1...
INFO:camkit3d.recorder:Camera 1 (Camera 1): 1280x720 @ 30fps
INFO:camkit3d.recorder:Connected to 2/2 cameras


{0: True, 1: True}

## Preview Cameras

In [12]:
recorder.preview_cameras(duration=15.0,target_fps=30)

INFO:camkit3d.recorder:Showing preview for 15.0s at ~30fps (press 'q' to quit)...
INFO:camkit3d.recorder:Preview resolution: 640x360 (50% of original)


## Record

In [4]:
timestamp = datetime.now().strftime("%Y%m%d_%H_%M_%S")
recorder.start_recording(f"{data_dir}recording_{timestamp}")

time.sleep(10)  # Record for 10 seconds
recorder.stop_recording()

INFO:camkit3d.recorder:Starting recording: /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25
INFO:camkit3d.recorder:Camera 0 recording to /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/raw_videos/camera_0.avi
INFO:camkit3d.recorder:Camera 1 recording to /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/raw_videos/camera_1.avi
INFO:camkit3d.recorder:Recording started on 2 cameras
INFO:camkit3d.recorder:Stopping recording...
INFO:camkit3d.recorder:Camera 0 stopped recording: 299 frames
INFO:camkit3d.recorder:Camera 1 stopped recording: 304 frames
INFO:camkit3d.recorder:Recording stopped: /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25


{0: (299,
  [1772798065.426647,
   1772798065.463122,
   1772798065.493155,
   1772798065.529963,
   1772798065.559564,
   1772798065.595475,
   1772798065.628062,
   1772798065.663278,
   1772798065.693161,
   1772798065.729702,
   1772798065.763108,
   1772798065.825505,
   1772798065.861015,
   1772798065.891831,
   1772798065.925617,
   1772798065.957406,
   1772798065.9913018,
   1772798066.023781,
   1772798066.0569038,
   1772798066.090552,
   1772798066.123272,
   1772798066.156938,
   1772798066.189705,
   1772798066.222251,
   1772798066.255829,
   1772798066.289074,
   1772798066.321686,
   1772798066.3553002,
   1772798066.385008,
   1772798066.421612,
   1772798066.454558,
   1772798066.487364,
   1772798066.5210052,
   1772798066.5539572,
   1772798066.586995,
   1772798066.6197991,
   1772798066.6531048,
   1772798066.68749,
   1772798066.717408,
   1772798066.7520251,
   1772798066.78548,
   1772798066.818538,
   1772798066.8515599,
   1772798066.884094,
   1772798066.9

## Disconnect Cameras

In [5]:
recorder.disconnect_cameras()

INFO:camkit3d.recorder:Disconnecting cameras...
INFO:camkit3d.recorder:Stopping camera 0...
INFO:camkit3d.recorder:Camera 0 stopped successfully
INFO:camkit3d.recorder:Stopping camera 1...
INFO:camkit3d.recorder:Camera 1 stopped successfully
INFO:camkit3d.recorder:All cameras disconnected successfully


## Synchronise

In [7]:
from camkit3d.sync import synchronize_videos_to_ideal_fps
from camkit3d.sync import plot_sync_results, plot_sync_summary_stats

In [8]:
# Synchronize all videos to camera 0
trial_dir = f"{data_dir}recording_{timestamp}"

# Synchronize to ideal 30 FPS (or whatever you specify)
results = synchronize_videos_to_ideal_fps(
    trial_folder=trial_dir,
    target_fps=30.0,  # Your target FPS
    max_time_diff_ms=50.0
)


Timestamp-based Synchronization to Ideal 30.0 FPS Clock
Trial folder: /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25
Raw videos:   /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/raw_videos
Output dir:   /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/synchronized_videos
Target FPS:   30.0

[1] Loading timestamps...
  Camera 1: 304 timestamps, range: [1772798065.430, 1772798075.544]s
  Camera 0: 299 timestamps, range: [1772798065.427, 1772798075.399]s

[2] Finding global time range...
  Global start time: 1772798065.430s
  Global end time:   1772798075.399s
  Duration:          9.969s

[3] Creating ideal 30.0 FPS timing grid...
  Ideal frame count: 300
  Ideal duration:    9.967s

[4] Finding video files...
  Camera 0: camera_0.avi
  Camera 1: camera_1.avi

[5] Getting video properties...
  Output FPS:        30.00
  Output resolution: 1280x720 (from camera 0)
  Output frames:     300

[6] Building frame mappings to id

## Plot the synchronisation results

In [11]:
# Create plots
figs = plot_sync_results(
    results=results,
    trial_folder=trial_dir,
    save_plots=True,
    show_plots=True
)

# Summary stats
fig_summary = plot_sync_summary_stats(
    results=results,
    trial_folder=trial_dir,
    save_plots=True,

)


📊 Plots saved to: /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/synchronization_plots/sync_analysis.png
📊 PDF saved to: /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/synchronization_plots/sync_analysis.pdf
📊 Summary plots saved to: /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/synchronization_plots/sync_summary.png


## Create synchronised videos side-by-side

In [12]:
from pathlib import Path
import math

import cv2
import numpy as np


def save_synced_videos_tiled(
    trial_folder,
    synced_subdir="synchronized_videos",
    out_name="synchronized_tiled.mp4",
    max_cols=3,
    max_width=1920,
    pad_px=6,
    pad_value=0,  # black padding
    font_scale=0.9,
    thickness=2,
):
    """
    Tile synchronized videos into a single review video (grid layout).

    - Auto-wrap: up to `max_cols` per row (default 3), then starts a new row.
    - Assumes synchronized and identical frame counts (still checks).
    - Input filenames: camera_<id>_synchronized.mp4

    Output is MP4 (H.264 if available, else mp4v).
    """
    trial_folder = Path(trial_folder)
    synced_dir = trial_folder / synced_subdir
    out_path = trial_folder / out_name

    videos = sorted(synced_dir.glob("camera_*_synchronized.mp4"))
    if not videos:
        raise FileNotFoundError(f"No synchronized videos found in {synced_dir}")

    caps = []
    cam_ids = []
    for v in videos:
        cap = cv2.VideoCapture(str(v))
        if not cap.isOpened():
            raise RuntimeError(f"Could not open {v}")
        caps.append(cap)
        cam_ids.append(int(v.stem.split("_")[1]))

    # --- checks ---
    frame_counts = [int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) for cap in caps]
    if len(set(frame_counts)) != 1:
        raise ValueError(f"Frame count mismatch: {dict(zip(cam_ids, frame_counts))}")

    fps = caps[0].get(cv2.CAP_PROP_FPS) or 25.0
    n_frames = frame_counts[0]

    # Determine a common cell size (max width/height across cameras)
    widths = [int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) for cap in caps]
    heights = [int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) for cap in caps]
    cell_w = max(widths)
    cell_h = max(heights)

    n = len(caps)
    cols = min(max_cols, n)
    rows = math.ceil(n / cols)

    grid_w = cols * cell_w + (cols - 1) * pad_px
    grid_h = rows * cell_h + (rows - 1) * pad_px

    # Optional downscale to fit screen width
    scale = 1.0
    if max_width and grid_w > max_width:
        scale = max_width / grid_w

    out_w = int(round(grid_w * scale))
    out_h = int(round(grid_h * scale))

    # Choose codec: try H.264 first (often works as 'avc1' on mac, 'H264' on some builds)
    def _make_writer(codec):
        fourcc = cv2.VideoWriter_fourcc(*codec)
        w = cv2.VideoWriter(str(out_path), fourcc, fps, (out_w, out_h))
        return w

    writer = _make_writer("avc1")
    if not writer.isOpened():
        writer = _make_writer("H264")
    if not writer.isOpened():
        writer = _make_writer("mp4v")
    if not writer.isOpened():
        raise RuntimeError(f"Could not open VideoWriter for {out_path}")

    print("Writing tiled video:")
    print(f"  cameras: {cam_ids}")
    print(f"  videos:  {n} -> grid {rows}x{cols}")
    print(f"  frames:  {n_frames}")
    print(f"  fps:     {fps:.2f}")
    print(f"  cell:    {cell_w}x{cell_h} px")
    print(f"  output:  {out_path}")

    # Reset all captures to frame 0
    for cap in caps:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # --- write ---
    for i in range(n_frames):
        # Start with a blank grid canvas
        canvas = np.full((grid_h, grid_w, 3), pad_value, dtype=np.uint8)

        for idx, (cam_id, cap) in enumerate(zip(cam_ids, caps)):
            ok, frame = cap.read()
            if not ok:
                raise RuntimeError(f"Failed reading frame {i} from camera {cam_id}")

            # Resize into cell (letterbox to preserve aspect ratio)
            fh, fw = frame.shape[:2]
            s = min(cell_w / fw, cell_h / fh)
            new_w = int(round(fw * s))
            new_h = int(round(fh * s))
            resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)

            cell = np.full((cell_h, cell_w, 3), pad_value, dtype=np.uint8)
            y0 = (cell_h - new_h) // 2
            x0 = (cell_w - new_w) // 2
            cell[y0:y0 + new_h, x0:x0 + new_w] = resized

            # overlay text (on the cell)
            cv2.putText(
                cell,
                f"Cam {cam_id} | frame {i}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                font_scale,
                (0, 255, 0),
                thickness,
                cv2.LINE_AA,
            )

            r = idx // cols
            c = idx % cols
            top = r * (cell_h + pad_px)
            left = c * (cell_w + pad_px)
            canvas[top:top + cell_h, left:left + cell_w] = cell

        # Final scale (if needed)
        if scale != 1.0:
            canvas = cv2.resize(canvas, (out_w, out_h), interpolation=cv2.INTER_AREA)

        writer.write(canvas)

    # cleanup
    writer.release()
    for cap in caps:
        cap.release()

    print("Done ✅")
    return out_path


In [13]:
# ---- usage ----
out_video = save_synced_videos_tiled(trial_dir)
out_video

Writing tiled video:
  cameras: [0, 1]
  videos:  2 -> grid 1x2
  frames:  300
  fps:     30.00
  cell:    1280x720 px
  output:  /Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/synchronized_tiled.mp4
Done ✅


PosixPath('/Users/robertseymour/Documents/recordings/recording_20260306_11_54_25/synchronized_tiled.mp4')